In [ ]:
'''
Use data from the literature to directly compare autaptic and synapse properties

'''

import numpy as np
import pandas as pandas
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# AQUA supplies the neuron/autapse model wrapper used to build Brian2 objects.
import aqua
from aqua.batchAQUA_general import batchAQUA

# Brian2 units, monitors, and network primitives are imported into the namespace.
from brian2 import *

In [ ]:
# import data 

filename = 'autapse_properties_updated.csv'
df = pd.read_csv(filename)
print(df.keys())

df['Study'] = df['Author'] + " (" + df['Year'].astype(str) + ")"

print(df.head())

In [ ]:
## Only look at the studies which look explicitly compare autapses to synapses.

# Clean up trailing whitespace in string columns if present
df['Author'] = df['Author'].str.strip()

# Convert autapse column values to standardized boolean/string representation
df['autapse_bool'] = df['autapse'].astype(str).str.upper()

# Filter groups that contain both 'TRUE' and 'FALSE'
filtered_df = df.groupby(['Author', 'Year']).filter(
    lambda group: {'TRUE', 'FALSE'}.issubset(set(group['autapse_bool']))
).drop(columns=['autapse_bool'])
filtered_df['Study'] = filtered_df['Author'] + " (" + filtered_df['Year'].astype(str) + ")"
# Print the resulting DataFrame
print(filtered_df)

In [ ]:
## SUPPLEMENTARY FIGURE
# 3. Measurement parameters
num_cols = [
    'peak current (pA)', 'CV aEPSC', 'failure rate', 
    'onset latency (ms)', 'rise time (ms)', 'decay time (ms)', 
    'probability of release', 'paired-pulse ratio'
]

def parse_val_and_std(val):
    """Parses cell string to extract numeric mean and standard deviation."""
    if pd.isna(val) or str(val).strip() in ['n/a', '']:
        return np.nan, np.nan
    
    clean_str = str(val).split('(')[0].strip()  # Strip frequency annotations like (20Hz)
    
    if '±' in clean_str:
        parts = clean_str.split('±')
        return float(parts[0].strip()), float(parts[1].strip())
    elif '+-' in clean_str:
        parts = clean_str.split('+-')
        return float(parts[0].strip()), float(parts[1].strip())
    else:
        try:
            return float(clean_str), np.nan
        except ValueError:
            return np.nan, np.nan

# Extract mean values (_val) and standard deviations (_std)
for col in num_cols:
    res = filtered_df[col].apply(parse_val_and_std)
    filtered_df[col + '_val'] = [r[0] for r in res]
    filtered_df[col + '_std'] = [r[1] for r in res]

# 4. Generate plots with custom _std error bars
fig, axes = plt.subplots(4, 2, figsize=(15, 18))
axes = axes.flatten()

sns.set_theme(style="whitegrid")
palette = {True: '#2b5c8f', False: '#d95f02'}
hue_order = [True, False]

for i, col in enumerate(num_cols):
    ax = axes[i]
    val_col = col
    std_col = col + '_std'
    
    sub_df = filtered_df.dropna(subset=[val_col])
    
    if not sub_df.empty:
        # Plot primary bars without auto-generated Seaborn error bars
        sns.barplot(
            data=sub_df, 
            x='Study', 
            y=val_col, 
            hue='autapse', 
            hue_order=hue_order,
            ax=ax,
            palette=palette,
            errorbar= None
        )

        
        # Calculate precise x positions to align error bars onto grouped bars
        studies = sub_df['Study'].unique()
        n_hues = len(hue_order)
        width = 0.8 / n_hues  # Standard bar width ratio in Seaborn
        
        for study_idx, study in enumerate(studies):
            for hue_idx, hue_val in enumerate(hue_order):
                row = sub_df[(sub_df['Study'] == study) & (sub_df['autapse'] == hue_val)]
                
                if not row.empty:
                    y_val = row[val_col].values[0]
                    y_err = row[std_col].values[0]
                    
                    # Plot error bar if _std exists and is valid
                    if not np.isnan(y_err):
                        x_pos = study_idx + (hue_idx - (n_hues - 1) / 2) * width
                        ax.errorbar(
                            x=x_pos, 
                            y=y_val, 
                            yerr=y_err, 
                            fmt='none', 
                            c='black', 
                            capsize=4, 
                            capthick=1.2, 
                            elinewidth=1.2
                        )
        
        ax.set_title(col, fontsize=12, fontweight='bold', pad=10)
        ax.set_xlabel('')
        ax.set_ylabel('Value', fontsize=10)
        ax.tick_params(axis='x', rotation=30)
        ax.legend(title='Autapse', frameon=True)
    else:
        ax.text(0.5, 0.5, 'No Data Available', horizontalalignment='center', 
                verticalalignment='center', transform=ax.transAxes, color='gray')
        ax.set_title(col, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('Supplementary Plots/autapse_custom_errorbars.png', dpi=300)
plt.show()